# 01 — Fine-Tuning de BETO (Spanish BERT) con GPU en Google Colab / Kaggle
**Autores:** Giuliano Crenna, Bruno Emmanuel Pace (UGR)  
**Etapa:** 4 — Modelado  
**Hardware requerido:** GPU dedicada (ej. Nvidia T4 en Google Colab gratuita o local)

Este notebook implementa el ajuste fino (fine-tuning) del modelo preentrenado **BETO** (`dccuchile/bert-base-spanish-wwm-cased`) para la clasificación de riesgo de depresión en textos en español.

## 1. Verificación de Hardware y GPU
En Google Colab: asegurate de ir a `Entorno de ejecución` -> `Cambiar tipo de entorno de ejecución` -> Seleccionar **GPU T4**.

In [ ]:
!nvidia-smi

## 2. Instalación de dependencias (sólo necesario en Colab)

In [ ]:
# Descomentar si se ejecuta en Google Colab:
# !pip install --upgrade transformers datasets accelerate evaluate pyarrow scikit-learn

## 3. Carga de librerías y configuración de reproducibilidad

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed
)
from sklearn.metrics import accuracy_score, f1_score, classification_report, roc_auc_score, cohen_kappa_score

SEED = 42
set_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Dispositivo en uso: {DEVICE.upper()}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 4. Carga de datos procesados (Splits de entrenamiento, validación y test)

In [ ]:
# Si estás en Google Colab con Google Drive montado:
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = Path('/content/drive/MyDrive/tesina/data/processed/splits')

DATA_DIR = Path("./data/processed/splits")

# Cargar parquet
df_train = pd.read_parquet(DATA_DIR / "train.parquet")
df_val = pd.read_parquet(DATA_DIR / "val.parquet")
df_test = pd.read_parquet(DATA_DIR / "test.parquet")

print(f"Train: {len(df_train):,} filas")
print(f"Val:   {len(df_val):,} filas")
print(f"Test:  {len(df_test):,} filas")

# Mapeo de etiquetas (0: control -> 0, 2: depresivo -> 1)
unique_labels = sorted(df_train["label"].unique())
label2id = {orig: idx for idx, orig in enumerate(unique_labels)}
id2label = {idx: orig for orig, idx in label2id.items()}
print(f"Mapeo de clases para PyTorch: {label2id}")

## 5. Tokenización con BETO y Dataset de PyTorch

In [ ]:
MODEL_NAME = "dccuchile/bert-base-spanish-wwm-cased"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = list(texts)
        self.labels = [label2id[int(y)] for y in labels]
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        item = {key: val.squeeze(0) for key, val in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_ds = TextDataset(df_train["text_clean"], df_train["label"], tokenizer, MAX_LENGTH)
val_ds = TextDataset(df_val["text_clean"], df_val["label"], tokenizer, MAX_LENGTH)
test_ds = TextDataset(df_test["text_clean"], df_test["label"], tokenizer, MAX_LENGTH)

## 6. Configuración del Modelo y Métricas

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label2id),
    id2label={str(k): str(v) for k, v in id2label.items()},
    label2id={str(k): str(v) for k, v in label2id.items()}
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    y_true = [id2label[y] for y in labels]
    y_pred = [id2label[p] for p in preds]
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1_weighted": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "kappa": float(cohen_kappa_score(y_true, y_pred))
    }

## 7. Fine-Tuning con Hugging Face Trainer (3 épocas, FP16, Early Stopping)

In [ ]:
training_args = TrainingArguments(
    output_dir="./beto_checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=100,
    save_total_limit=1,
    seed=SEED,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()

## 8. Evaluación en Test y Guardado de Resultados

In [ ]:
test_results = trainer.evaluate(eval_dataset=test_ds)
print("=== RESULTADOS FINALES EN TEST ===")
for k, v in test_results.items():
    print(f"{k}: {v}")

# Guardar modelo final y tokenizer
OUT_MODEL_PATH = Path("./models/beto/best_model")
OUT_MODEL_PATH.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(OUT_MODEL_PATH))
tokenizer.save_pretrained(str(OUT_MODEL_PATH))
print(f"Modelo y tokenizer exportados a {OUT_MODEL_PATH}")